# 10 - Agregación de señales gastronómicas

## Objetivo

Este notebook construye el primer sistema de agregación de señales para Hidden Gems.

Partimos del archivo:

`dish_mentions_with_sentiment_hybrid_v1.jsonl`

Ese archivo contiene menciones de platos ya detectadas, normalizadas y enriquecidas con sentimiento por mención.

El objetivo de este módulo es transformar menciones individuales en señales agregadas:

```text
mención individual
→ plato normalizado
→ sentimiento
→ confianza
→ fiabilidad
→ agregación por plato
→ agregación por plato + negocio

```
## Agregaciones principales

Se generarán dos tablas principales:

dish_global_signals_v1.csv

Agregación global por plato:

pizza
burger
fries
sushi
tacos
...
dish_business_signals_v1.csv

Agregación por plato dentro de cada negocio:

business_id + dish_id

Esta segunda tabla será fundamental para el ranking posterior.

## Métricas calculadas

Se calcularán señales como:

- número de menciones;
- número de reviews;
- número de negocios;
- menciones positivas, neutrales y negativas;
- ratio positivo;
- ratio negativo;
- sentimiento medio;
- sentimiento ponderado por confianza;
- menciones de alta fiabilidad;
- confianza media del NER;
- confianza media del sentimiento;
- señales mínimas para ranking.


## Salidas esperadas

dish_global_signals_v1.csv
dish_business_signals_v1.csv
dish_signal_aggregation_summary_v1.json
muestras de revisión para ranking posterior

In [1]:
# ============================================================
# 01. Imports y configuración base
# ============================================================

from pathlib import Path
import json
import random
import re
import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 350)

print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


In [2]:
# ============================================================
# 02. Detectar entorno y preparar carpetas
# ============================================================

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IS_KAGGLE:
    ENV_NAME = "kaggle"
elif IS_COLAB:
    ENV_NAME = "colab"
else:
    ENV_NAME = "local"

print("Entorno detectado:", ENV_NAME)

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "hidden_gems_ai"

elif IS_COLAB:
    from google.colab import drive

    try:
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")

    except Exception as e:
        print("No se ha podido montar Drive. Se usará almacenamiento temporal.")
        print("Error:", e)

        PROJECT_DIR = Path("/content/hidden_gems_ai")

else:
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs" / "dish_signal_aggregation"
SIGNALS_DIR = OUTPUT_DIR / "signals"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
SAMPLES_DIR = OUTPUT_DIR / "samples"

for path in [
    PROJECT_DIR,
    OUTPUT_DIR,
    SIGNALS_DIR,
    ARTIFACTS_DIR,
    SAMPLES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Entorno detectado: kaggle
PROJECT_DIR: /kaggle/working/hidden_gems_ai
OUTPUT_DIR: /kaggle/working/hidden_gems_ai/outputs/dish_signal_aggregation


In [3]:
# ============================================================
# 03. Diagnóstico de archivos disponibles
# ============================================================

if IS_KAGGLE:
    print("Archivos disponibles en /kaggle/input:")

    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("No estás en Kaggle. Se buscarán archivos en el proyecto local.")

Archivos disponibles en /kaggle/input:
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_labels.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_high_precision_with_negatives.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_normalization_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_manual_review_sample_v2_150.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/transformer_sentiment_metrics.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_with_sentiment_hybrid_v1.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidates_v2.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_inference_summary_full.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_exploration_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-g

In [4]:
# ============================================================
# 04. Localizar archivo de menciones con sentimiento
# ============================================================

MENTIONS_SENTIMENT_FILENAME = "dish_mentions_with_sentiment_hybrid_v1.jsonl"
SENTIMENT_SUMMARY_FILENAME = "dish_mention_sentiment_summary_hybrid_v1.json"

def find_file(filename: str) -> Path:
    candidate_paths = []

    if IS_KAGGLE:
        candidate_paths.extend(list(Path("/kaggle/input").rglob(filename)))

    candidate_paths.extend(list(PROJECT_DIR.rglob(filename)))
    candidate_paths.extend(list(Path.cwd().rglob(filename)))

    candidate_paths = list(dict.fromkeys(candidate_paths))

    if not candidate_paths:
        raise FileNotFoundError(
            f"No se ha encontrado el archivo requerido: {filename}\n"
            "En Kaggle, asegúrate de haberlo añadido desde Add Data."
        )

    return candidate_paths[0]


MENTIONS_SENTIMENT_PATH = find_file(MENTIONS_SENTIMENT_FILENAME)

try:
    SENTIMENT_SUMMARY_PATH = find_file(SENTIMENT_SUMMARY_FILENAME)
except FileNotFoundError:
    SENTIMENT_SUMMARY_PATH = None

print("MENTIONS_SENTIMENT_PATH:", MENTIONS_SENTIMENT_PATH)
print("SENTIMENT_SUMMARY_PATH:", SENTIMENT_SUMMARY_PATH)

MENTIONS_SENTIMENT_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_with_sentiment_hybrid_v1.jsonl
SENTIMENT_SUMMARY_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mention_sentiment_summary_hybrid_v1.json


In [5]:
# ============================================================
# 05. Funciones auxiliares de carga y guardado
# ============================================================

def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    invalid_json_count = 0

    with path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Leyendo {path.name}"):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                invalid_json_count += 1

    print(f"Archivo: {path.name}")
    print(f"Registros cargados: {len(records):,}")
    print(f"Líneas JSON inválidas: {invalid_json_count:,}")

    return pd.DataFrame(records)


def make_json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, set):
        return sorted(list(value))

    return value


def save_jsonl(df_to_save: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in df_to_save.to_dict(orient="records"):
            clean_record = {
                str(key): make_json_safe(value)
                for key, value in record.items()
            }

            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

    print(f"Guardado: {path} ({len(df_to_save):,} registros)")

In [6]:
# ============================================================
# 06. Cargar menciones con sentimiento
# ============================================================

mentions_raw_df = load_jsonl(MENTIONS_SENTIMENT_PATH)

print("Shape:", mentions_raw_df.shape)

print("\nColumnas:")
print(mentions_raw_df.columns.tolist())

display(mentions_raw_df.head(5))

if SENTIMENT_SUMMARY_PATH is not None:
    with SENTIMENT_SUMMARY_PATH.open("r", encoding="utf-8") as f:
        sentiment_summary = json.load(f)

    print("\nResumen del módulo 09:")
    print(json.dumps(sentiment_summary, indent=2, ensure_ascii=False)[:4000])

Leyendo dish_mentions_with_sentiment_hybrid_v1.jsonl: 0it [00:00, ?it/s]

Archivo: dish_mentions_with_sentiment_hybrid_v1.jsonl
Registros cargados: 94,932
Líneas JSON inválidas: 0
Shape: (94932, 48)

Columnas:
['mention_id', 'corpus_document_id', 'review_id', 'business_id', 'business_name', 'city', 'state', 'split', 'language', 'source_system_code', 'source_dataset', 'dish_mention_text', 'dish_mention_normalized', 'canonical_dish_name_v2', 'dish_id_v2', 'normalization_status_v2', 'normalization_method_v2', 'start_char', 'end_char', 'start_token', 'end_token', 'token_count', 'confidence_mean', 'confidence_min', 'confidence_max', 'rating_value', 'sentiment_label', 'review_text', 'sentence_context', 'window_context', 'target_clause_context_v11', 'near_mention_context_v11', 'mention_sentiment_label_final', 'mention_sentiment_score_final', 'mention_sentiment_confidence_final', 'sentiment_reliability_tier', 'sentiment_reason_final', 'sentiment_flags_final', 'sentiment_method_final', 'positive_terms_v11', 'negative_terms_v11', 'target_signal_strength_v11', 'rating_

,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,canonical_dish_name_v2,dish_id_v2,normalization_status_v2,normalization_method_v2,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,rating_value,sentiment_label,review_text,sentence_context,window_context,target_clause_context_v11,near_mention_context_v11,mention_sentiment_label_final,mention_sentiment_score_final,mention_sentiment_confidence_final,sentiment_reliability_tier,sentiment_reason_final,sentiment_flags_final,sentiment_method_final,positive_terms_v11,negative_terms_v11,target_signal_strength_v11,rating_prior_score_v11,review_prior_score_v11,direct_positive_count_v11,direct_negative_count_v11,is_sentiment_training_candidate,human_review_status
0,dish_mention_c6a233f28ab0ec50a4bf,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,en,yelp_open_dataset,yelp_open_dataset,tamale,tamale,tamale,dish_seed_v2_000060,normalized_rule_based_v2,rule_based_v2_overrides,88,94,18,19,1,0.997417,0.997417,0.997417,3.0,neutral,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with no expectations. Next to the Clarion Hotel.","Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served a","Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","rtment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has",positive,1.1500,0.7066,medium,target_context_primary,[],hybrid_context_lexicon_v1_1,"[fresh:positive_lexicon, good:positive_lexicon]",[],1.8,0.00,0.00,0,0,False,not_reviewed
1,dish_mention_370e33f760028e4e87a3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,en,yelp_open_dataset,yelp_open_dataset,lamb curry,lamb curry,lamb curry,dish_seed_v2_000205,normalized_rule_based_v2,rule_based_v2_overrides,54,64,12,14,2,0.996373,0.993416,0.999331,5.0,positive,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",Our favorite is the lamb curry and korma.,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...g",Our favorite is the lamb curry and korma.,"y, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we",positive,1.2130,0.7128,medium,target_context_primary,[],hybrid_context_lexicon_v1_1,"[delicious:positive_lexicon, favorite:positive_lexicon]",[],1.8,0.75,0.35,0,0,False,not_reviewed
2,dish_mention_2c51d464763786d9fef2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,en,yelp_open_dataset,yelp_open_dataset,korma,korma,korma,dish_seed_v2_000074,normalized_rule_based_v2,rule_based_v2_overrides,69,74,15,16,1,0.997466,0.997466,0.997466,5.0,positive,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something ne


Resumen del módulo 09:
{
  "notebook": "09_dish_mention_sentiment_hybrid_v1",
  "version": "hybrid_context_lexicon_v1_1",
  "source_mentions_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_normalized_v2.jsonl",
  "input": {
    "total_mentions_for_sentiment": 94932,
    "unique_reviews": 42461,
    "unique_businesses": 4088,
    "unique_dishes": 9937,
    "unique_canonical_dish_names": 9937,
    "confidence": {
      "min": 0.35621654987335205,
      "mean": 0.97570959051333,
      "median": 0.9990743398666382,
      "max": 0.9997578263282776
    },
    "rating_counts": {
      "1.0": 7363,
      "2.0": 9032,
      "3.0": 13904,
      "4.0": 29471,
      "5.0": 35162
    },
    "review_sentiment_counts": {
      "positive": 64633,
      "negative": 16395,
      "neutral": 13904
    },
    "split_counts": {
      "train": 75951,
      "validation": 9546,
      "test": 9435
    },
    "top_dishes": {
      "pizza": 8751,
      "burger": 8095,
      "

In [7]:
# ============================================================
# 07. Validación de columnas necesarias
# ============================================================

required_cols = [
    "mention_id",
    "review_id",
    "business_id",
    "dish_id_v2",
    "canonical_dish_name_v2",
    "dish_mention_text",
    "confidence_mean",
    "rating_value",
    "sentiment_label",
    "mention_sentiment_label_final",
    "mention_sentiment_score_final",
    "mention_sentiment_confidence_final",
    "sentiment_reliability_tier",
    "sentiment_reason_final",
    "is_sentiment_training_candidate",
]

missing_cols = [
    col for col in required_cols
    if col not in mentions_raw_df.columns
]

if missing_cols:
    raise ValueError(f"Faltan columnas obligatorias: {missing_cols}")

print("Columnas obligatorias presentes.")

print("\nNulos principales:")
display(
    mentions_raw_df[required_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nDuplicados mention_id:")
print(mentions_raw_df["mention_id"].duplicated().sum())

Columnas obligatorias presentes.

Nulos principales:


mention_id                            0
review_id                             0
business_id                           0
dish_id_v2                            0
canonical_dish_name_v2                0
dish_mention_text                     0
confidence_mean                       0
rating_value                          0
sentiment_label                       0
mention_sentiment_label_final         0
mention_sentiment_score_final         0
mention_sentiment_confidence_final    0
sentiment_reliability_tier            0
sentiment_reason_final                0
is_sentiment_training_candidate       0
dtype: int64


Duplicados mention_id:
0


In [8]:
# ============================================================
# 08. Preparar dataframe base para agregación
# ============================================================

mentions_df = mentions_raw_df.copy()

# IDs
mentions_df["mention_id"] = mentions_df["mention_id"].astype(str)
mentions_df["review_id"] = mentions_df["review_id"].astype(str)
mentions_df["business_id"] = mentions_df["business_id"].astype(str)
mentions_df["dish_id_v2"] = mentions_df["dish_id_v2"].astype(str)

# Texto de plato
mentions_df["canonical_dish_name_v2"] = (
    mentions_df["canonical_dish_name_v2"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
    .str.strip()
)

mentions_df["dish_mention_text"] = (
    mentions_df["dish_mention_text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Numéricos
numeric_cols = [
    "confidence_mean",
    "confidence_min",
    "confidence_max",
    "rating_value",
    "mention_sentiment_score_final",
    "mention_sentiment_confidence_final",
]

for col in numeric_cols:
    if col in mentions_df.columns:
        mentions_df[col] = pd.to_numeric(mentions_df[col], errors="coerce")

# Categóricos
mentions_df["mention_sentiment_label_final"] = (
    mentions_df["mention_sentiment_label_final"]
    .astype(str)
    .str.lower()
    .str.strip()
)

mentions_df["sentiment_reliability_tier"] = (
    mentions_df["sentiment_reliability_tier"]
    .astype(str)
    .str.lower()
    .str.strip()
)

mentions_df["sentiment_label"] = (
    mentions_df["sentiment_label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

if "split" in mentions_df.columns:
    mentions_df["split"] = mentions_df["split"].astype(str).str.lower().str.strip()
else:
    mentions_df["split"] = "unknown"

# Limpieza de válidos
valid_sentiment_labels = {"positive", "neutral", "negative"}
valid_reliability_tiers = {"high", "medium", "low"}

before_count = len(mentions_df)

mentions_df = mentions_df[
    mentions_df["dish_id_v2"].notna()
    & (mentions_df["dish_id_v2"].str.len() > 0)
    & mentions_df["canonical_dish_name_v2"].notna()
    & (mentions_df["canonical_dish_name_v2"].str.len() > 0)
    & mentions_df["mention_sentiment_label_final"].isin(valid_sentiment_labels)
    & mentions_df["sentiment_reliability_tier"].isin(valid_reliability_tiers)
    & mentions_df["mention_sentiment_score_final"].notna()
    & mentions_df["mention_sentiment_confidence_final"].notna()
].copy()

after_count = len(mentions_df)

print("Menciones antes:", before_count)
print("Menciones válidas para agregación:", after_count)
print("Descartadas:", before_count - after_count)

display(mentions_df.head(5))

Menciones antes: 94932
Menciones válidas para agregación: 94932
Descartadas: 0


,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,canonical_dish_name_v2,dish_id_v2,normalization_status_v2,normalization_method_v2,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,rating_value,sentiment_label,review_text,sentence_context,window_context,target_clause_context_v11,near_mention_context_v11,mention_sentiment_label_final,mention_sentiment_score_final,mention_sentiment_confidence_final,sentiment_reliability_tier,sentiment_reason_final,sentiment_flags_final,sentiment_method_final,positive_terms_v11,negative_terms_v11,target_signal_strength_v11,rating_prior_score_v11,review_prior_score_v11,direct_positive_count_v11,direct_negative_count_v11,is_sentiment_training_candidate,human_review_status
0,dish_mention_c6a233f28ab0ec50a4bf,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,en,yelp_open_dataset,yelp_open_dataset,tamale,tamale,tamale,dish_seed_v2_000060,normalized_rule_based_v2,rule_based_v2_overrides,88,94,18,19,1,0.997417,0.997417,0.997417,3.0,neutral,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with no expectations. Next to the Clarion Hotel.","Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served a","Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","rtment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has",positive,1.1500,0.7066,medium,target_context_primary,[],hybrid_context_lexicon_v1_1,"[fresh:positive_lexicon, good:positive_lexicon]",[],1.8,0.00,0.00,0,0,False,not_reviewed
1,dish_mention_370e33f760028e4e87a3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,en,yelp_open_dataset,yelp_open_dataset,lamb curry,lamb curry,lamb curry,dish_seed_v2_000205,normalized_rule_based_v2,rule_based_v2_overrides,54,64,12,14,2,0.996373,0.993416,0.999331,5.0,positive,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",Our favorite is the lamb curry and korma.,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...g",Our favorite is the lamb curry and korma.,"y, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we",positive,1.2130,0.7128,medium,target_context_primary,[],hybrid_context_lexicon_v1_1,"[delicious:positive_lexicon, favorite:positive_lexicon]",[],1.8,0.75,0.35,0,0,False,not_reviewed
2,dish_mention_2c51d464763786d9fef2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,en,yelp_open_dataset,yelp_open_dataset,korma,korma,korma,dish_seed_v2_000074,normalized_rule_based_v2,rule_based_v2_overrides,69,74,15,16,1,0.997466,0.997466,0.997466,5.0,positive,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something ne

In [9]:
# ============================================================
# 09. Crear pesos y señales numéricas
# ============================================================

RELIABILITY_WEIGHT_MAP = {
    "high": 1.00,
    "medium": 0.65,
    "low": 0.25,
}

SENTIMENT_LABEL_SCORE_MAP = {
    "positive": 1.0,
    "neutral": 0.0,
    "negative": -1.0,
}

mentions_df["reliability_weight"] = mentions_df["sentiment_reliability_tier"].map(
    RELIABILITY_WEIGHT_MAP
).fillna(0.25)

mentions_df["sentiment_label_score"] = mentions_df["mention_sentiment_label_final"].map(
    SENTIMENT_LABEL_SCORE_MAP
).fillna(0.0)

# Score híbrido normalizado a [-1, 1].
# El score del módulo 09 está aproximadamente en [-5, 5].
mentions_df["sentiment_score_normalized"] = (
    mentions_df["mention_sentiment_score_final"] / 5.0
).clip(-1.0, 1.0)

# Peso total de señal: fiabilidad * confianza del sentimiento * confianza NER.
mentions_df["ner_confidence"] = mentions_df["confidence_mean"].fillna(1.0).clip(0.0, 1.0)
mentions_df["sentiment_confidence"] = mentions_df["mention_sentiment_confidence_final"].fillna(0.0).clip(0.0, 1.0)

mentions_df["signal_weight"] = (
    mentions_df["reliability_weight"]
    * mentions_df["sentiment_confidence"]
    * mentions_df["ner_confidence"]
)

# Señal final combinada: usamos etiqueta y score continuo.
mentions_df["weighted_sentiment_value"] = (
    0.65 * mentions_df["sentiment_label_score"]
    + 0.35 * mentions_df["sentiment_score_normalized"]
)

mentions_df["weighted_sentiment_value"] = mentions_df["weighted_sentiment_value"].clip(-1.0, 1.0)

mentions_df["weighted_sentiment_contribution"] = (
    mentions_df["weighted_sentiment_value"]
    * mentions_df["signal_weight"]
)

mentions_df["positive_signal"] = np.where(
    mentions_df["mention_sentiment_label_final"] == "positive",
    mentions_df["signal_weight"],
    0.0
)

mentions_df["negative_signal"] = np.where(
    mentions_df["mention_sentiment_label_final"] == "negative",
    mentions_df["signal_weight"],
    0.0
)

mentions_df["neutral_signal"] = np.where(
    mentions_df["mention_sentiment_label_final"] == "neutral",
    mentions_df["signal_weight"],
    0.0
)

mentions_df["is_high_reliability"] = mentions_df["sentiment_reliability_tier"] == "high"
mentions_df["is_medium_reliability"] = mentions_df["sentiment_reliability_tier"] == "medium"
mentions_df["is_low_reliability"] = mentions_df["sentiment_reliability_tier"] == "low"

print("Distribución de signal_weight:")
display(mentions_df["signal_weight"].describe())

print("\nDistribución de weighted_sentiment_value:")
display(mentions_df["weighted_sentiment_value"].describe())

display(
    mentions_df[
        [
            "dish_mention_text",
            "canonical_dish_name_v2",
            "mention_sentiment_label_final",
            "mention_sentiment_score_final",
            "sentiment_score_normalized",
            "sentiment_reliability_tier",
            "signal_weight",
            "weighted_sentiment_value",
            "weighted_sentiment_contribution",
        ]
    ].head(10)
)

Distribución de signal_weight:


count    94932.000000
mean         0.377814
std          0.336725
min          0.020312
25%          0.071769
50%          0.362497
75%          0.753798
max          0.949770
Name: signal_weight, dtype: float64


Distribución de weighted_sentiment_value:


count    94932.000000
mean         0.311703
std          0.480739
min         -1.000000
25%          0.013090
50%          0.041755
75%          0.752410
max          1.000000
Name: weighted_sentiment_value, dtype: float64

,dish_mention_text,canonical_dish_name_v2,mention_sentiment_label_final,mention_sentiment_score_final,sentiment_score_normalized,sentiment_reliability_tier,signal_weight,weighted_sentiment_value,weighted_sentiment_contribution
0,tamale,tamale,positive,1.1500,0.2300,medium,0.458103,0.730500,0.334645
1,lamb curry,lamb curry,positive,1.2130,0.2426,medium,0.461640,0.734910,0.339264
2,korma,korma,positive,0.9630,0.1926,medium,0.435693,0.717410,0.312571
3,naan,naan,neutral,0.3465,0.0693,low,0.071801,0.024255,0.001742
4,sandwiches,sandwich,positive,0.8545,0.1709,medium,0.427074,0.709815,0.303143
5,wings,wings,positive,1.7130,0.3426,high,0.794477,0.769910,0.611675
6,sushi,sushi,neutral,0.0000,0.0000,low,0.064969,0.000000,0.000000
7,gnocchi with marinara,gnocchi with marinara,positive,5.0000,1.0000,high,0.734242,1.000000,0.734242
8,baked eggplant appetizer,baked eggplant appetizer,positive,4.8670,0.9734,medium,0.426894,0.990690,0.422920
9,Sonoran Dog,sonoran dog,neutral,0.2310,0.0462,low,0.068467,0.016170,0.001107


In [10]:
# ============================================================
# 10. QA inicial de señales
# ============================================================

aggregation_input_qa = {
    "total_mentions": int(len(mentions_df)),
    "unique_reviews": int(mentions_df["review_id"].nunique()),
    "unique_businesses": int(mentions_df["business_id"].nunique()),
    "unique_dishes": int(mentions_df["dish_id_v2"].nunique()),
    "sentiment_counts": {
        str(k): int(v)
        for k, v in mentions_df["mention_sentiment_label_final"].value_counts().to_dict().items()
    },
    "reliability_counts": {
        str(k): int(v)
        for k, v in mentions_df["sentiment_reliability_tier"].value_counts().to_dict().items()
    },
    "signal_weight": {
        "min": float(mentions_df["signal_weight"].min()),
        "mean": float(mentions_df["signal_weight"].mean()),
        "median": float(mentions_df["signal_weight"].median()),
        "max": float(mentions_df["signal_weight"].max()),
    },
    "weighted_sentiment_value": {
        "min": float(mentions_df["weighted_sentiment_value"].min()),
        "mean": float(mentions_df["weighted_sentiment_value"].mean()),
        "median": float(mentions_df["weighted_sentiment_value"].median()),
        "max": float(mentions_df["weighted_sentiment_value"].max()),
    },
}

print(json.dumps(aggregation_input_qa, indent=2, ensure_ascii=False))

print("\nTop platos por menciones:")
display(
    mentions_df["canonical_dish_name_v2"]
    .value_counts()
    .head(50)
    .reset_index()
    .rename(columns={
        "index": "canonical_dish_name_v2",
        "canonical_dish_name_v2": "mention_count"
    })
)

{
  "total_mentions": 94932,
  "unique_reviews": 42461,
  "unique_businesses": 4088,
  "unique_dishes": 9937,
  "sentiment_counts": {
    "neutral": 45366,
    "positive": 43142,
    "negative": 6424
  },
  "reliability_counts": {
    "low": 44426,
    "medium": 25495,
    "high": 25011
  },
  "signal_weight": {
    "min": 0.020311710990965366,
    "mean": 0.37781376200795724,
    "median": 0.3624973218077422,
    "max": 0.9497699350118637
  },
  "weighted_sentiment_value": {
    "min": -1.0,
    "mean": 0.3117034681140184,
    "median": 0.041755,
    "max": 1.0
  }
}

Top platos por menciones:


,mention_count,count
0,pizza,8751
1,burger,8095
2,fries,5375
3,sushi,4470
4,tacos,4436
5,shrimp,4421
6,steak,2794
7,ice cream,2411
8,wings,2215
9,donut,2175


## 1. Agregación global por plato

En esta fase se agregan todas las menciones por `dish_id_v2`.

Esta tabla permite responder preguntas como:

- ¿Qué platos aparecen más?
- ¿Qué platos tienen mejor señal media?
- ¿Qué platos tienen más menciones positivas?
- ¿Qué platos tienen señales más fiables?

La tabla global todavía no es el ranking Hidden Gems definitivo. Es una tabla de señales base.

In [11]:
# ============================================================
# 11. Función auxiliar de agregación
# ============================================================

def safe_weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = ~np.isnan(values) & ~np.isnan(weights) & (weights > 0)

    if mask.sum() == 0:
        return np.nan

    return float(np.average(values[mask], weights=weights[mask]))


def compute_aggregate_quality(row) -> str:
    """
    Clasifica la calidad de una agregación.
    """

    mention_count = row.get("mention_count", 0)
    review_count = row.get("review_count", 0)
    high_reliability_mentions = row.get("high_reliability_mentions", 0)
    avg_signal_weight = row.get("avg_signal_weight", 0.0)

    if mention_count >= 50 and review_count >= 25 and high_reliability_mentions >= 10 and avg_signal_weight >= 0.35:
        return "high"

    if mention_count >= 15 and review_count >= 8 and avg_signal_weight >= 0.20:
        return "medium"

    return "low"


def compute_positive_ratio(row):
    denominator = row["positive_mentions"] + row["neutral_mentions"] + row["negative_mentions"]
    if denominator == 0:
        return 0.0
    return float(row["positive_mentions"] / denominator)


def compute_negative_ratio(row):
    denominator = row["positive_mentions"] + row["neutral_mentions"] + row["negative_mentions"]
    if denominator == 0:
        return 0.0
    return float(row["negative_mentions"] / denominator)

In [12]:
# ============================================================
# 12. Agregación global por plato
# ============================================================

global_group_cols = [
    "dish_id_v2",
    "canonical_dish_name_v2",
]

dish_global_basic_df = (
    mentions_df
    .groupby(global_group_cols, dropna=False)
    .agg(
        mention_count=("mention_id", "size"),
        review_count=("review_id", "nunique"),
        business_count=("business_id", "nunique"),

        positive_mentions=("mention_sentiment_label_final", lambda s: int((s == "positive").sum())),
        neutral_mentions=("mention_sentiment_label_final", lambda s: int((s == "neutral").sum())),
        negative_mentions=("mention_sentiment_label_final", lambda s: int((s == "negative").sum())),

        high_reliability_mentions=("is_high_reliability", "sum"),
        medium_reliability_mentions=("is_medium_reliability", "sum"),
        low_reliability_mentions=("is_low_reliability", "sum"),

        avg_ner_confidence=("ner_confidence", "mean"),
        avg_sentiment_confidence=("sentiment_confidence", "mean"),
        avg_signal_weight=("signal_weight", "mean"),
        total_signal_weight=("signal_weight", "sum"),

        avg_rating=("rating_value", "mean"),
        avg_sentiment_score_raw=("mention_sentiment_score_final", "mean"),
        avg_sentiment_score_normalized=("sentiment_score_normalized", "mean"),
        avg_weighted_sentiment_value=("weighted_sentiment_value", "mean"),

        positive_signal=("positive_signal", "sum"),
        neutral_signal=("neutral_signal", "sum"),
        negative_signal=("negative_signal", "sum"),
        weighted_sentiment_contribution=("weighted_sentiment_contribution", "sum"),
    )
    .reset_index()
)

# Weighted sentiment aggregate
weighted_rows = []

for row in tqdm(
    dish_global_basic_df.to_dict(orient="records"),
    desc="Calculando weighted sentiment global"
):
    dish_id = row["dish_id_v2"]

    subset = mentions_df[mentions_df["dish_id_v2"] == dish_id]

    weighted_mean = safe_weighted_mean(
        subset["weighted_sentiment_value"],
        subset["signal_weight"]
    )

    weighted_rows.append({
        "dish_id_v2": dish_id,
        "confidence_weighted_sentiment": weighted_mean,
    })

global_weighted_df = pd.DataFrame(weighted_rows)

dish_global_signals_df = dish_global_basic_df.merge(
    global_weighted_df,
    on="dish_id_v2",
    how="left"
)

dish_global_signals_df["positive_ratio"] = dish_global_signals_df.apply(
    compute_positive_ratio,
    axis=1
)

dish_global_signals_df["negative_ratio"] = dish_global_signals_df.apply(
    compute_negative_ratio,
    axis=1
)

dish_global_signals_df["reliability_high_ratio"] = (
    dish_global_signals_df["high_reliability_mentions"]
    / dish_global_signals_df["mention_count"]
).fillna(0.0)

dish_global_signals_df["aggregate_quality_tier"] = dish_global_signals_df.apply(
    compute_aggregate_quality,
    axis=1
)

dish_global_signals_df = dish_global_signals_df.sort_values(
    ["mention_count", "confidence_weighted_sentiment"],
    ascending=[False, False]
).reset_index(drop=True)

print("Platos globales agregados:", len(dish_global_signals_df))

display(dish_global_signals_df.head(50))

Calculando weighted sentiment global:   0%|          | 0/9937 [00:00<?, ?it/s]

Platos globales agregados: 9937


,dish_id_v2,canonical_dish_name_v2,mention_count,review_count,business_count,positive_mentions,neutral_mentions,negative_mentions,high_reliability_mentions,medium_reliability_mentions,low_reliability_mentions,avg_ner_confidence,avg_sentiment_confidence,avg_signal_weight,total_signal_weight,avg_rating,avg_sentiment_score_raw,avg_sentiment_score_normalized,avg_weighted_sentiment_value,positive_signal,neutral_signal,negative_signal,weighted_sentiment_contribution,confidence_weighted_sentiment,positive_ratio,negative_ratio,reliability_high_ratio,aggregate_quality_tier
0,dish_seed_v2_000001,pizza,8751,4599,856,3779,4392,580,2120,2405,4226,0.997650,0.536322,0.367837,3218.940705,3.648269,0.755191,0.151038,0.290476,2436.995169,437.060041,344.885495,1695.956618,0.526868,0.431836,0.066278,0.242258,high
1,dish_seed_v2_000002,burger,8095,4478,863,3564,3963,568,2071,2256,3768,0.994412,0.544419,0.378229,3061.762930,3.685361,0.778708,0.155742,0.295078,2314.414949,402.890142,344.457840,1599.673682,0.522468,0.440272,0.070167,0.255837,high
2,dish_seed_v2_000003,fries,5375,4094,1093,2421,2466,488,1705,1348,2322,0.997493,0.572823,0.420975,2262.741152,3.600744,0.824837,0.164967,0.291497,1676.142129,269.823090,316.775933,1120.752702,0.495308,0.450419,0.090791,0.317209,high
3,dish_seed_v2_000004,sushi,4470,2482,266,2028,2171,271,1125,1250,2095,0.999147,0.543060,0.376949,1684.962116,3.833781,0.809885,0.161977,0.312184,1309.028935,215.565349,160.367832,928.652208,0.551141,0.453691,0.060626,0.251678,high
4,dish_seed_v2_000005,tacos,4436,2707,533,1859,2274,303,1076,1113,2247,0.995108,0.528524,0.359207,1593.443117,3.783138,0.775993,0.155199,0.282318,1202.246045,221.020725,170.176347,843.924612,0.529623,0.419071,0.068305,0.242561,high
5,dish_seed_v2_000006,shrimp,4421,3269,865,2022,2057,342,1259,1086,2076,0.993723,0.554314,0.392569,1735.547668,3.902285,0.878616,0.175723,0.308506,1322.279291,205.058687,208.209689,919.380469,0.529735,0.457363,0.077358,0.284777,high
6,dish_seed_v2_000007,steak,2794,2102,830,1095,1410,289,765,667,1362,0.994526,0.541866,0.379903,1061.449278,3.556908,0.642151,0.128430,0.232460,730.532019,142.315091,188.602168,445.381380,0.419597,0.391911,0.103436,0.273801,high
7,dish_seed_v2_000008,ice cream,2411,1588,475,1048,1283,80,495,647,1269,0.999356,0.511730,0.334994,807.671738,4.096640,0.831030,0.166206,0.319143,647.558194,119.489486,40.624058,486.698209,0.602594,0.434674,0.033181,0.205309,medium
8,dish_seed_v2_000009,wings,2215,1493,588,1053,1002,160,655,590,970,0.995117,0.567206,0.409473,906.982334,3.597743,0.861874,0.172375,0.322385,699.882576,105.385614,101.714145,490.275996,0.540557,0.475395,0.072235,0.295711,high
9,dish_seed_v2_000010,donut,2175,973,175,881,1171,123,477,567,1131,0.997435,0.516398,0.343593,747.315529,4.061149,0.745292,0.149058,0.278699,559.389931,113.962448,73.963150,391.187438,0.523457,0.405057,0.056552,0.219310,medium


## 2. Agregación por plato + negocio

Esta tabla será la base más importante para el ranking Hidden Gems.

A nivel global, `pizza` o `burger` pueden ser muy frecuentes, pero Hidden Gems necesita saber:

```text
en qué negocio concreto se habla especialmente bien de un plato concreto
```
Por eso se agregan señales por:
```
business_id + dish_id_v2
```
Ejemplo:
```
Local A + ramen
Local B + burger
Local C + fried chicken
```

In [13]:
# ============================================================
# 13. Agregación por negocio y plato
# ============================================================

business_group_cols = [
    "business_id",
    "dish_id_v2",
    "canonical_dish_name_v2",
]

optional_business_cols = [
    "business_name",
    "city",
    "state",
]

for col in optional_business_cols:
    if col not in mentions_df.columns:
        mentions_df[col] = ""

business_info_df = (
    mentions_df
    .groupby("business_id", dropna=False)
    .agg(
        business_name=("business_name", "first"),
        city=("city", "first"),
        state=("state", "first"),
    )
    .reset_index()
)

dish_business_basic_df = (
    mentions_df
    .groupby(business_group_cols, dropna=False)
    .agg(
        mention_count=("mention_id", "size"),
        review_count=("review_id", "nunique"),

        positive_mentions=("mention_sentiment_label_final", lambda s: int((s == "positive").sum())),
        neutral_mentions=("mention_sentiment_label_final", lambda s: int((s == "neutral").sum())),
        negative_mentions=("mention_sentiment_label_final", lambda s: int((s == "negative").sum())),

        high_reliability_mentions=("is_high_reliability", "sum"),
        medium_reliability_mentions=("is_medium_reliability", "sum"),
        low_reliability_mentions=("is_low_reliability", "sum"),

        avg_ner_confidence=("ner_confidence", "mean"),
        avg_sentiment_confidence=("sentiment_confidence", "mean"),
        avg_signal_weight=("signal_weight", "mean"),
        total_signal_weight=("signal_weight", "sum"),

        avg_rating=("rating_value", "mean"),
        avg_sentiment_score_raw=("mention_sentiment_score_final", "mean"),
        avg_sentiment_score_normalized=("sentiment_score_normalized", "mean"),
        avg_weighted_sentiment_value=("weighted_sentiment_value", "mean"),

        positive_signal=("positive_signal", "sum"),
        neutral_signal=("neutral_signal", "sum"),
        negative_signal=("negative_signal", "sum"),
        weighted_sentiment_contribution=("weighted_sentiment_contribution", "sum"),
    )
    .reset_index()
)

weighted_business_rows = []

for row in tqdm(
    dish_business_basic_df.to_dict(orient="records"),
    desc="Calculando weighted sentiment por negocio-plato"
):
    business_id = row["business_id"]
    dish_id = row["dish_id_v2"]

    subset = mentions_df[
        (mentions_df["business_id"] == business_id)
        & (mentions_df["dish_id_v2"] == dish_id)
    ]

    weighted_mean = safe_weighted_mean(
        subset["weighted_sentiment_value"],
        subset["signal_weight"]
    )

    weighted_business_rows.append({
        "business_id": business_id,
        "dish_id_v2": dish_id,
        "confidence_weighted_sentiment": weighted_mean,
    })

business_weighted_df = pd.DataFrame(weighted_business_rows)

dish_business_signals_df = dish_business_basic_df.merge(
    business_weighted_df,
    on=["business_id", "dish_id_v2"],
    how="left"
)

dish_business_signals_df = dish_business_signals_df.merge(
    business_info_df,
    on="business_id",
    how="left"
)

dish_business_signals_df["positive_ratio"] = dish_business_signals_df.apply(
    compute_positive_ratio,
    axis=1
)

dish_business_signals_df["negative_ratio"] = dish_business_signals_df.apply(
    compute_negative_ratio,
    axis=1
)

dish_business_signals_df["reliability_high_ratio"] = (
    dish_business_signals_df["high_reliability_mentions"]
    / dish_business_signals_df["mention_count"]
).fillna(0.0)

dish_business_signals_df["aggregate_quality_tier"] = dish_business_signals_df.apply(
    compute_aggregate_quality,
    axis=1
)

# Señal mínima para que luego pueda entrar en ranking.
dish_business_signals_df["is_rankable_candidate_v1"] = (
    (dish_business_signals_df["mention_count"] >= 2)
    & (dish_business_signals_df["review_count"] >= 2)
    & (dish_business_signals_df["total_signal_weight"] >= 0.75)
)

dish_business_signals_df = dish_business_signals_df.sort_values(
    [
        "is_rankable_candidate_v1",
        "confidence_weighted_sentiment",
        "mention_count",
        "positive_ratio",
    ],
    ascending=[False, False, False, False]
).reset_index(drop=True)

print("Pares negocio-plato agregados:", len(dish_business_signals_df))
print("Candidatos rankeables v1:", int(dish_business_signals_df["is_rankable_candidate_v1"].sum()))

display(dish_business_signals_df.head(50))

Calculando weighted sentiment por negocio-plato:   0%|          | 0/31036 [00:00<?, ?it/s]

Pares negocio-plato agregados: 31036
Candidatos rankeables v1: 6375


,business_id,dish_id_v2,canonical_dish_name_v2,mention_count,review_count,positive_mentions,neutral_mentions,negative_mentions,high_reliability_mentions,medium_reliability_mentions,low_reliability_mentions,avg_ner_confidence,avg_sentiment_confidence,avg_signal_weight,total_signal_weight,avg_rating,avg_sentiment_score_raw,avg_sentiment_score_normalized,avg_weighted_sentiment_value,positive_signal,neutral_signal,negative_signal,weighted_sentiment_contribution,confidence_weighted_sentiment,business_name,city,state,positive_ratio,negative_ratio,reliability_high_ratio,aggregate_quality_tier,is_rankable_candidate_v1
0,kxX2SOes4o-D3ZQBkiMRfA,dish_seed_v2_001268,makhani,2,2,2,0,0,1,1,0,0.824138,0.935850,0.646765,1.293530,5.000000,5.000000,1.000000,1.000000,1.293530,0.000000,0.0,1.293530,1.000000,Zaika,Philadelphia,PA,1.000000,0.0,0.500000,low,True
1,Kzf6VUWTKtSBd4OWUfwlkg,dish_seed_v2_000002,burger,2,2,2,0,0,2,0,0,0.997551,0.950000,0.947673,1.895347,5.000000,4.806500,0.961300,0.986455,1.895347,0.000000,0.0,1.869622,0.986427,Frosty Boy Drive-In,New Palestine,IN,1.000000,0.0,1.000000,low,True
2,ena3aLdMz2ym_OPVuTIJ2g,dish_seed_v2_000014,crab,2,2,2,0,0,2,0,0,0.999470,0.950000,0.949497,1.898993,4.500000,4.796000,0.959200,0.985720,1.898993,0.000000,0.0,1.871873,0.985719,Broadway Bar & Grille,Clifton Heights,PA,1.000000,0.0,1.000000,low,True
3,DeVlppoc8dPBhOCPrm4wmg,dish_seed_v2_000237,wedge salad,2,2,2,0,0,1,0,1,0.800147,0.774000,0.517928,1.035856,4.000000,2.906500,0.581300,0.853455,1.035856,0.000000,0.0,1.009340,0.974402,Harry & Izzy's,Indianapolis,IN,1.000000,0.0,0.500000,low,True
4,i-tDq8zC7ZmSqSbg_7oddA,dish_seed_v2_000062,salad,3,2,3,0,0,1,1,1,0.851320,0.680200,0.433360,1.300080,5.000000,3.453167,0.690633,0.891722,1.300080,0.000000,0.0,1.266214,0.973951,LA Smokehouse,New Orleans,LA,1.000000,0.0,0.333333,low,True
5,O3Gsne_JJ24yMvLrDoNjqA,dish_seed_v2_000074,korma,2,2,2,0,0,2,0,0,0.996777,0.950000,0.946938,1.893877,4.500000,4.577500,0.915500,0.970425,1.893877,0.000000,0.0,1.837906,0.970446,Nirvana Fusion,Hatboro,PA,1.000000,0.0,1.000000,low,True
6,SZU9c8V2GuREDN5KgyHFJw,dish_seed_v2_000194,lobster bisque,2,2,2,0,0,2,0,0,0.896303,0.945350,0.847541,1.695081,4.500000,4.519000,0.903800,0.966330,1.695081,0.000000,0.0,1.641310,0.968278,Santa Barbara Shellfish Company,Santa Barbara,CA,1.000000,0.0,1.000000,low,True
7,z22hSRptt_DS0nWjsIka2A,dish_seed_v2_000018,ribs,3,2,2,1,0,2,0,1,0.999211,0.726667,0.656167,1.968500,3.666667,3.416667,0.683333,0.672500,1.898560,0.069940,0.0,1.899784,0.965092,Outback Steakhouse,Royersford,PA,0.666667,0.0,0.666667,low,True
8,xwKYBPO0ByGlkvNcr8FdqQ,dish_seed_v2_000007,steak,4,2,4,0,0,4,0,0,0.998727,0.950000,0.948790,3.795162,5.000000,4.472125,0.894425,0.963049,3.795162,0.000000,0.0,3.654883,0.963037,Le Rendez-vous,Tucson,AZ,1.000000,0.0,1.000000,low,True
9,ew_Hhp12Silh3qjoPaW9IA,dish_seed_v2_000035,dumplings,2,2,2,0,0,2,0,0,0.999450,0.894950,0.894461,1.788921,5.000000,4.375500,0.875100,0.956285,1.788921,0.000000,0.0,1.714477,0.958386,Thai Legacy Restaurant,Brandon,FL,1.000000,0.0,1.000000,low,True


In [14]:
# ============================================================
# 14. QA de agregaciones
# ============================================================

aggregation_summary = {
    "notebook": "10_dish_signal_aggregation",
    "version": "signal_aggregation_v1",
    "source_mentions_path": str(MENTIONS_SENTIMENT_PATH),

    "input": aggregation_input_qa,

    "global_signals": {
        "total_dishes": int(len(dish_global_signals_df)),
        "quality_tier_counts": {
            str(k): int(v)
            for k, v in dish_global_signals_df["aggregate_quality_tier"].value_counts().to_dict().items()
        },
        "top_by_mentions": (
            dish_global_signals_df
            .sort_values("mention_count", ascending=False)
            .head(30)
            [
                [
                    "canonical_dish_name_v2",
                    "mention_count",
                    "review_count",
                    "business_count",
                    "positive_ratio",
                    "negative_ratio",
                    "confidence_weighted_sentiment",
                    "aggregate_quality_tier",
                ]
            ]
            .to_dict(orient="records")
        ),
        "top_by_weighted_sentiment_min_20_mentions": (
            dish_global_signals_df[
                dish_global_signals_df["mention_count"] >= 20
            ]
            .sort_values("confidence_weighted_sentiment", ascending=False)
            .head(30)
            [
                [
                    "canonical_dish_name_v2",
                    "mention_count",
                    "review_count",
                    "business_count",
                    "positive_ratio",
                    "negative_ratio",
                    "confidence_weighted_sentiment",
                    "aggregate_quality_tier",
                ]
            ]
            .to_dict(orient="records")
        ),
    },

    "business_dish_signals": {
        "total_business_dish_pairs": int(len(dish_business_signals_df)),
        "rankable_candidates_v1": int(dish_business_signals_df["is_rankable_candidate_v1"].sum()),
        "quality_tier_counts": {
            str(k): int(v)
            for k, v in dish_business_signals_df["aggregate_quality_tier"].value_counts().to_dict().items()
        },
        "top_rankable_candidates": (
            dish_business_signals_df[
                dish_business_signals_df["is_rankable_candidate_v1"]
            ]
            .sort_values(
                ["confidence_weighted_sentiment", "mention_count", "positive_ratio"],
                ascending=[False, False, False]
            )
            .head(50)
            [
                [
                    "business_id",
                    "business_name",
                    "city",
                    "state",
                    "canonical_dish_name_v2",
                    "mention_count",
                    "review_count",
                    "positive_ratio",
                    "negative_ratio",
                    "confidence_weighted_sentiment",
                    "total_signal_weight",
                    "aggregate_quality_tier",
                ]
            ]
            .to_dict(orient="records")
        ),
    }
}

print(json.dumps(aggregation_summary, indent=2, ensure_ascii=False)[:7000])

print("\nGlobal quality tiers:")
display(dish_global_signals_df["aggregate_quality_tier"].value_counts())

print("\nBusiness-dish quality tiers:")
display(dish_business_signals_df["aggregate_quality_tier"].value_counts())

print("\nTop global por sentimiento ponderado, mínimo 20 menciones:")
display(
    dish_global_signals_df[
        dish_global_signals_df["mention_count"] >= 20
    ]
    .sort_values("confidence_weighted_sentiment", ascending=False)
    .head(30)
    [
        [
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "business_count",
            "positive_ratio",
            "negative_ratio",
            "confidence_weighted_sentiment",
            "aggregate_quality_tier",
        ]
    ]
)

{
  "notebook": "10_dish_signal_aggregation",
  "version": "signal_aggregation_v1",
  "source_mentions_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_with_sentiment_hybrid_v1.jsonl",
  "input": {
    "total_mentions": 94932,
    "unique_reviews": 42461,
    "unique_businesses": 4088,
    "unique_dishes": 9937,
    "sentiment_counts": {
      "neutral": 45366,
      "positive": 43142,
      "negative": 6424
    },
    "reliability_counts": {
      "low": 44426,
      "medium": 25495,
      "high": 25011
    },
    "signal_weight": {
      "min": 0.020311710990965366,
      "mean": 0.37781376200795724,
      "median": 0.3624973218077422,
      "max": 0.9497699350118637
    },
    "weighted_sentiment_value": {
      "min": -1.0,
      "mean": 0.3117034681140184,
      "median": 0.041755,
      "max": 1.0
    }
  },
  "global_signals": {
    "total_dishes": 9937,
    "quality_tier_counts": {
      "low": 9782,
      "medium": 95,
      "high": 60
    }

aggregate_quality_tier
low       9782
medium      95
high        60
Name: count, dtype: int64


Business-dish quality tiers:


aggregate_quality_tier
low       30219
medium      698
high        119
Name: count, dtype: int64


Top global por sentimiento ponderado, mínimo 20 menciones:


,canonical_dish_name_v2,mention_count,review_count,business_count,positive_ratio,negative_ratio,confidence_weighted_sentiment,aggregate_quality_tier
124,lobster risotto,20,19,6,0.700000,0.000000,0.803143,medium
85,caesar salad,41,38,36,0.658537,0.024390,0.789225,medium
97,bbq shrimp,31,30,9,0.741935,0.000000,0.772352,medium
94,brisket ramen,32,28,2,0.593750,0.000000,0.765963,medium
75,coffee,68,68,53,0.779412,0.058824,0.752304,high
116,curry,23,22,21,0.521739,0.043478,0.743579,medium
90,greek salad,34,32,28,0.470588,0.029412,0.739871,medium
99,tempura shrimp,30,28,18,0.566667,0.000000,0.739072,medium
95,lobster mac and cheese,32,28,12,0.656250,0.031250,0.738933,medium
103,sonoran,27,26,8,0.777778,0.000000,0.736071,medium


In [15]:
# ============================================================
# 15. Guardar outputs del primer bloque
# ============================================================

global_signals_path = SIGNALS_DIR / "dish_global_signals_v1.csv"
business_signals_path = SIGNALS_DIR / "dish_business_signals_v1.csv"
summary_path = ARTIFACTS_DIR / "dish_signal_aggregation_summary_v1.json"

dish_global_signals_df.to_csv(global_signals_path, index=False)
dish_business_signals_df.to_csv(business_signals_path, index=False)

aggregation_summary["outputs"] = {
    "dish_global_signals": str(global_signals_path),
    "dish_business_signals": str(business_signals_path),
    "summary": str(summary_path),
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(aggregation_summary, f, indent=2, ensure_ascii=False)

print("Outputs guardados:")
print("global_signals_path:", global_signals_path)
print("business_signals_path:", business_signals_path)
print("summary_path:", summary_path)

Outputs guardados:
global_signals_path: /kaggle/working/hidden_gems_ai/outputs/dish_signal_aggregation/signals/dish_global_signals_v1.csv
business_signals_path: /kaggle/working/hidden_gems_ai/outputs/dish_signal_aggregation/signals/dish_business_signals_v1.csv
summary_path: /kaggle/working/hidden_gems_ai/outputs/dish_signal_aggregation/artifacts/dish_signal_aggregation_summary_v1.json


## 3. Scoring inicial para ranking

La agregación por negocio-plato ya está creada, pero todavía no debe usarse directamente como ranking.

Problemas detectados:

1. Algunos pares tienen sentimiento muy alto, pero solo 2 menciones.
2. Algunos nombres canónicos siguen siendo ruidosos o demasiado extendidos.
3. El ranking necesita penalizar baja evidencia.
4. El ranking necesita separar candidatos fuertes, emergentes y no listos.

En este bloque se añade una capa de scoring preliminar:

- control de calidad del nombre del plato;
- tier de evidencia;
- sentimiento bayesiano con prior neutral;
- penalización por baja evidencia;
- penalización por ruido;
- score preliminar para ranking.

Este score todavía no incorpora barrios. Será la base para el siguiente notebook de ranking.

In [16]:
# ============================================================
# 16. Reglas de calidad para nombres de platos
# ============================================================

DISH_NAME_EXACT_NOISE = {
    "",
    "#",
    "up",
    "food",
    "meal",
    "menu",
    "dish",
    "dishes",
    "side",
    "sides",
    "appetizer",
    "appetizers",
    "starter",
    "starters",
    "entree",
    "entrees",
    "drink",
    "drinks",
    "water",
    "sauce",
    "dressing",
    "good",
    "great",
    "was",
    "were",
    "ordered",
    "order",
    "service",
    "place",
    "restaurant",
}

SUSPICIOUS_WORDS_IN_DISH_NAME = {
    "wife",
    "husband",
    "server",
    "waiter",
    "waitress",
    "price",
    "size",
    "ordered",
    "order",
    "came",
    "come",
    "served",
    "was",
    "were",
    "is",
    "are",
    "very",
    "really",
    "good",
    "great",
    "bad",
}

POSSIBLE_BEVERAGE_TERMS = {
    "coffee",
    "latte",
    "tea",
    "smoothie",
    "mojito",
    "beer",
    "wine",
    "cocktail",
    "margarita",
    "espresso",
    "cappuccino",
}

def build_dish_name_quality_flags(name: str) -> list[str]:
    flags = []

    if not isinstance(name, str):
        return ["invalid_name"]

    clean = re.sub(r"\s+", " ", name.lower()).strip()
    words = clean.split()

    if not clean:
        flags.append("empty_name")

    if clean in DISH_NAME_EXACT_NOISE:
        flags.append("exact_noise_name")

    if len(clean) <= 2 and clean not in {"bbq"}:
        flags.append("too_short_name")

    if len(clean) > 70:
        flags.append("too_long_name")

    if len(words) > 7:
        flags.append("too_many_words")

    if re.search(r"[:#@<>]", clean):
        flags.append("suspicious_punctuation")

    if re.search(r"\d", clean):
        flags.append("contains_digit")

    if words:
        if words[0] in {"was", "were", "is", "are", "the", "a", "an"}:
            flags.append("bad_start_word")

        if words[-1] in {"was", "were", "is", "are", "with", "and", "or", "the", "a", "an"}:
            flags.append("bad_end_word")

    suspicious_hits = sorted(set(words) & SUSPICIOUS_WORDS_IN_DISH_NAME)

    if suspicious_hits:
        flags.append("contains_suspicious_words")

    if clean in POSSIBLE_BEVERAGE_TERMS:
        flags.append("possible_beverage")

    return sorted(set(flags))


def is_hard_noise_dish_name(flags: list[str]) -> bool:
    hard_flags = {
        "empty_name",
        "invalid_name",
        "exact_noise_name",
        "too_short_name",
        "too_long_name",
        "too_many_words",
        "suspicious_punctuation",
        "contains_digit",
        "bad_start_word",
        "bad_end_word",
    }

    return any(flag in hard_flags for flag in flags)


for df_name, df_obj in [
    ("dish_global_signals_df", dish_global_signals_df),
    ("dish_business_signals_df", dish_business_signals_df),
]:
    df_obj["dish_name_quality_flags"] = df_obj["canonical_dish_name_v2"].apply(
        build_dish_name_quality_flags
    )

    df_obj["is_hard_noise_dish_name"] = df_obj["dish_name_quality_flags"].apply(
        is_hard_noise_dish_name
    )

    df_obj["is_possible_beverage"] = df_obj["dish_name_quality_flags"].apply(
        lambda flags: "possible_beverage" in flags
    )

    print(df_name)
    print("hard noise:", int(df_obj["is_hard_noise_dish_name"].sum()))
    print("possible beverage:", int(df_obj["is_possible_beverage"].sum()))
    print()

dish_global_signals_df
hard noise: 696
possible beverage: 8

dish_business_signals_df
hard noise: 806
possible beverage: 359



In [17]:
# ============================================================
# 17. Tiers de evidencia
# ============================================================

def compute_business_evidence_tier(row) -> str:
    mention_count = int(row.get("mention_count", 0))
    review_count = int(row.get("review_count", 0))
    total_signal_weight = float(row.get("total_signal_weight", 0.0))
    high_reliability_mentions = int(row.get("high_reliability_mentions", 0))

    if (
        mention_count >= 10
        and review_count >= 8
        and total_signal_weight >= 4.0
        and high_reliability_mentions >= 3
    ):
        return "strong"

    if (
        mention_count >= 5
        and review_count >= 4
        and total_signal_weight >= 2.0
    ):
        return "solid"

    if (
        mention_count >= 3
        and review_count >= 3
        and total_signal_weight >= 1.25
    ):
        return "emerging"

    return "weak"


def compute_global_evidence_tier(row) -> str:
    mention_count = int(row.get("mention_count", 0))
    review_count = int(row.get("review_count", 0))
    business_count = int(row.get("business_count", 0))
    total_signal_weight = float(row.get("total_signal_weight", 0.0))

    if (
        mention_count >= 500
        and review_count >= 250
        and business_count >= 50
        and total_signal_weight >= 100
    ):
        return "strong"

    if (
        mention_count >= 100
        and review_count >= 50
        and business_count >= 15
        and total_signal_weight >= 25
    ):
        return "solid"

    if (
        mention_count >= 20
        and review_count >= 10
        and business_count >= 3
        and total_signal_weight >= 5
    ):
        return "emerging"

    return "weak"


dish_business_signals_df["business_dish_evidence_tier"] = dish_business_signals_df.apply(
    compute_business_evidence_tier,
    axis=1
)

dish_global_signals_df["global_evidence_tier"] = dish_global_signals_df.apply(
    compute_global_evidence_tier,
    axis=1
)

print("Business-dish evidence tiers:")
display(dish_business_signals_df["business_dish_evidence_tier"].value_counts())

print("\nGlobal evidence tiers:")
display(dish_global_signals_df["global_evidence_tier"].value_counts())

Business-dish evidence tiers:


business_dish_evidence_tier
weak        27194
emerging     1440
solid        1406
strong        996
Name: count, dtype: int64


Global evidence tiers:


global_evidence_tier
weak        9815
emerging      55
strong        34
solid         33
Name: count, dtype: int64

In [18]:
# ============================================================
# 18. Scoring conservador negocio-plato
# ============================================================

PRIOR_SIGNAL_WEIGHT = 3.0
NEUTRAL_SENTIMENT_PRIOR = 0.0

business_scoring_df = dish_business_signals_df.copy()

# Sentimiento bayesiano con prior neutral.
# Evita que pares con 2 menciones y score perfecto dominen el ranking.
business_scoring_df["bayesian_sentiment_score"] = (
    (
        business_scoring_df["confidence_weighted_sentiment"].fillna(0.0)
        * business_scoring_df["total_signal_weight"].fillna(0.0)
    )
    + (NEUTRAL_SENTIMENT_PRIOR * PRIOR_SIGNAL_WEIGHT)
) / (
    business_scoring_df["total_signal_weight"].fillna(0.0)
    + PRIOR_SIGNAL_WEIGHT
)

business_scoring_df["bayesian_sentiment_component"] = (
    (business_scoring_df["bayesian_sentiment_score"] + 1.0) / 2.0
).clip(0.0, 1.0)

business_scoring_df["evidence_score"] = (
    np.log1p(business_scoring_df["review_count"])
    / np.log1p(12)
).clip(0.0, 1.0)

business_scoring_df["mention_volume_score"] = (
    np.log1p(business_scoring_df["mention_count"])
    / np.log1p(15)
).clip(0.0, 1.0)

business_scoring_df["signal_weight_score"] = (
    business_scoring_df["total_signal_weight"] / 5.0
).clip(0.0, 1.0)

business_scoring_df["confidence_quality_score"] = (
    business_scoring_df["avg_signal_weight"] / 0.55
).clip(0.0, 1.0)

business_scoring_df["positive_balance_score"] = (
    business_scoring_df["positive_ratio"]
    - business_scoring_df["negative_ratio"]
).clip(-1.0, 1.0)

business_scoring_df["positive_balance_component"] = (
    (business_scoring_df["positive_balance_score"] + 1.0) / 2.0
).clip(0.0, 1.0)

business_scoring_df["negative_penalty_factor"] = (
    1.0 - (business_scoring_df["negative_ratio"] * 1.25)
).clip(0.55, 1.0)

business_scoring_df["noise_penalty_factor"] = np.where(
    business_scoring_df["is_hard_noise_dish_name"],
    0.0,
    1.0
)

business_scoring_df["beverage_penalty_factor"] = np.where(
    business_scoring_df["is_possible_beverage"],
    0.85,
    1.0
)

business_scoring_df["preliminary_ranking_score_v1"] = 100.0 * (
    0.34 * business_scoring_df["bayesian_sentiment_component"]
    + 0.18 * business_scoring_df["positive_balance_component"]
    + 0.18 * business_scoring_df["evidence_score"]
    + 0.12 * business_scoring_df["mention_volume_score"]
    + 0.10 * business_scoring_df["signal_weight_score"]
    + 0.08 * business_scoring_df["reliability_high_ratio"].clip(0.0, 1.0)
)

business_scoring_df["preliminary_ranking_score_v1"] = (
    business_scoring_df["preliminary_ranking_score_v1"]
    * business_scoring_df["negative_penalty_factor"]
    * business_scoring_df["noise_penalty_factor"]
    * business_scoring_df["beverage_penalty_factor"]
)

business_scoring_df["preliminary_ranking_score_v1"] = (
    business_scoring_df["preliminary_ranking_score_v1"]
    .clip(0.0, 100.0)
)

business_scoring_df["is_ranking_ready_v1"] = (
    ~business_scoring_df["is_hard_noise_dish_name"]
    & business_scoring_df["business_dish_evidence_tier"].isin({"emerging", "solid", "strong"})
    & (business_scoring_df["total_signal_weight"] >= 1.25)
    & (business_scoring_df["review_count"] >= 3)
    & (business_scoring_df["mention_count"] >= 3)
)

def assign_preliminary_candidate_tier(row) -> str:
    if not row["is_ranking_ready_v1"]:
        return "not_ready"

    score = float(row["preliminary_ranking_score_v1"])
    evidence_tier = row["business_dish_evidence_tier"]

    if score >= 75 and evidence_tier in {"solid", "strong"}:
        return "strong_candidate"

    if score >= 65 and evidence_tier in {"emerging", "solid", "strong"}:
        return "promising_candidate"

    if score >= 55:
        return "weak_candidate"

    return "not_ready"


business_scoring_df["preliminary_candidate_tier_v1"] = business_scoring_df.apply(
    assign_preliminary_candidate_tier,
    axis=1
)

business_scoring_df = business_scoring_df.sort_values(
    [
        "is_ranking_ready_v1",
        "preliminary_ranking_score_v1",
        "business_dish_evidence_tier",
        "review_count",
        "mention_count",
    ],
    ascending=[False, False, True, False, False]
).reset_index(drop=True)

print("Pares negocio-plato:", len(business_scoring_df))
print("Ranking ready v1:", int(business_scoring_df["is_ranking_ready_v1"].sum()))

print("\nCandidate tiers:")
display(business_scoring_df["preliminary_candidate_tier_v1"].value_counts())

display(
    business_scoring_df[
        [
            "business_name",
            "city",
            "state",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "positive_ratio",
            "negative_ratio",
            "confidence_weighted_sentiment",
            "bayesian_sentiment_score",
            "total_signal_weight",
            "business_dish_evidence_tier",
            "preliminary_ranking_score_v1",
            "preliminary_candidate_tier_v1",
            "dish_name_quality_flags",
        ]
    ].head(50)
)

Pares negocio-plato: 31036
Ranking ready v1: 3841

Candidate tiers:


preliminary_candidate_tier_v1
not_ready              28147
weak_candidate          1223
promising_candidate     1021
strong_candidate         645
Name: count, dtype: int64

,business_name,city,state,canonical_dish_name_v2,mention_count,review_count,positive_ratio,negative_ratio,confidence_weighted_sentiment,bayesian_sentiment_score,total_signal_weight,business_dish_evidence_tier,preliminary_ranking_score_v1,preliminary_candidate_tier_v1,dish_name_quality_flags
0,The Little Lamb,Clearwater,FL,burger,15,13,0.866667,0.000000,0.802149,0.600349,8.924912,strong,88.805930,strong_candidate,[]
1,East Beach Grill,Santa Barbara,CA,pancake,24,14,0.791667,0.000000,0.804999,0.665284,14.285097,strong,88.768158,strong_candidate,[]
2,Three Muses,New Orleans,LA,steak,32,26,0.781250,0.000000,0.776507,0.663566,17.626015,strong,88.061879,strong_candidate,[]
3,Sushi Ushi,Valrico,FL,sushi,53,28,0.716981,0.000000,0.770510,0.694509,27.414415,strong,87.580242,strong_candidate,[]
4,4 Rivers Smokehouse,Tampa Bay,FL,brisket,16,13,0.750000,0.000000,0.783810,0.606483,10.260428,strong,87.560215,strong_candidate,[]
5,Goody's Soda Fountain & Candy Store,Boise,ID,ice cream,30,20,0.666667,0.000000,0.778703,0.657213,16.228838,strong,87.439295,strong_candidate,[]
6,Backspace Bar & Kitchen,New Orleans,LA,burger,18,14,0.777778,0.000000,0.824746,0.640074,10.398027,strong,87.436818,strong_candidate,[]
7,Barbecue and Bourbon,Speedway,IN,pulled pork,12,12,0.833333,0.000000,0.754434,0.553283,8.251728,strong,87.340456,strong_candidate,[]
8,Empress Garden,Philadelphia,PA,pancake,15,14,0.866667,0.000000,0.818223,0.602247,8.365439,strong,87.238194,strong_candidate,[]
9,Plaza Deli,Santa Barbara,CA,sandwich,19,16,0.789474,0.000000,0.781770,0.607592,10.464968,strong,87.223795,strong_candidate,[]


In [19]:
# ============================================================
# 19. Scoring global por plato
# ============================================================

global_scoring_df = dish_global_signals_df.copy()

global_scoring_df["global_sentiment_component"] = (
    (global_scoring_df["confidence_weighted_sentiment"].fillna(0.0) + 1.0) / 2.0
).clip(0.0, 1.0)

global_scoring_df["global_evidence_score"] = (
    np.log1p(global_scoring_df["review_count"])
    / np.log1p(500)
).clip(0.0, 1.0)

global_scoring_df["global_business_coverage_score"] = (
    np.log1p(global_scoring_df["business_count"])
    / np.log1p(200)
).clip(0.0, 1.0)

global_scoring_df["global_positive_balance_score"] = (
    global_scoring_df["positive_ratio"]
    - global_scoring_df["negative_ratio"]
).clip(-1.0, 1.0)

global_scoring_df["global_positive_balance_component"] = (
    (global_scoring_df["global_positive_balance_score"] + 1.0) / 2.0
).clip(0.0, 1.0)

global_scoring_df["global_negative_penalty_factor"] = (
    1.0 - (global_scoring_df["negative_ratio"] * 1.25)
).clip(0.55, 1.0)

global_scoring_df["global_noise_penalty_factor"] = np.where(
    global_scoring_df["is_hard_noise_dish_name"],
    0.0,
    1.0
)

global_scoring_df["global_beverage_penalty_factor"] = np.where(
    global_scoring_df["is_possible_beverage"],
    0.85,
    1.0
)

global_scoring_df["global_ranking_signal_score_v1"] = 100.0 * (
    0.35 * global_scoring_df["global_sentiment_component"]
    + 0.20 * global_scoring_df["global_positive_balance_component"]
    + 0.20 * global_scoring_df["global_evidence_score"]
    + 0.15 * global_scoring_df["global_business_coverage_score"]
    + 0.10 * global_scoring_df["reliability_high_ratio"].clip(0.0, 1.0)
)

global_scoring_df["global_ranking_signal_score_v1"] = (
    global_scoring_df["global_ranking_signal_score_v1"]
    * global_scoring_df["global_negative_penalty_factor"]
    * global_scoring_df["global_noise_penalty_factor"]
    * global_scoring_df["global_beverage_penalty_factor"]
).clip(0.0, 100.0)

global_scoring_df = global_scoring_df.sort_values(
    [
        "global_ranking_signal_score_v1",
        "global_evidence_tier",
        "mention_count",
    ],
    ascending=[False, True, False]
).reset_index(drop=True)

print("Global scoring generado:", len(global_scoring_df))

display(
    global_scoring_df[
        [
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "business_count",
            "positive_ratio",
            "negative_ratio",
            "confidence_weighted_sentiment",
            "global_evidence_tier",
            "global_ranking_signal_score_v1",
            "dish_name_quality_flags",
        ]
    ].head(50)
)

Global scoring generado: 9937


,canonical_dish_name_v2,mention_count,review_count,business_count,positive_ratio,negative_ratio,confidence_weighted_sentiment,global_evidence_tier,global_ranking_signal_score_v1,dish_name_quality_flags
0,sandwich,2127,1851,820,0.521392,0.036201,0.647257,strong,77.937467,[]
1,hummus,632,503,179,0.506329,0.042722,0.629934,strong,76.536377,[]
2,ice cream,2411,1588,475,0.434674,0.033181,0.602594,strong,75.832065,[]
3,eggplant,381,329,197,0.475066,0.036745,0.613559,solid,75.487664,[]
4,french toast,877,718,203,0.500570,0.051311,0.606910,strong,75.388615,[]
5,pancake,2035,1314,267,0.452580,0.048157,0.586884,strong,74.633747,[]
6,spring rolls,418,361,141,0.555024,0.055024,0.618937,solid,74.485805,[]
7,pulled pork,601,529,203,0.492512,0.058236,0.575957,strong,74.114866,[]
8,gnocchi,251,202,86,0.561753,0.031873,0.649598,solid,73.816951,[]
9,waffle,879,646,226,0.422071,0.048919,0.559584,strong,73.735905,[]


In [20]:
# ============================================================
# 20. QA del scoring preliminar
# ============================================================

scoring_summary = {
    "notebook": "10_dish_signal_aggregation",
    "version": "signal_scoring_v1",
    "source_mentions_path": str(MENTIONS_SENTIMENT_PATH),

    "input": aggregation_input_qa,

    "business_dish_scoring": {
        "total_pairs": int(len(business_scoring_df)),
        "ranking_ready_v1": int(business_scoring_df["is_ranking_ready_v1"].sum()),
        "candidate_tier_counts": {
            str(k): int(v)
            for k, v in business_scoring_df["preliminary_candidate_tier_v1"].value_counts().to_dict().items()
        },
        "evidence_tier_counts": {
            str(k): int(v)
            for k, v in business_scoring_df["business_dish_evidence_tier"].value_counts().to_dict().items()
        },
        "hard_noise_pairs": int(business_scoring_df["is_hard_noise_dish_name"].sum()),
        "possible_beverage_pairs": int(business_scoring_df["is_possible_beverage"].sum()),
        "score_summary": {
            "min": float(business_scoring_df["preliminary_ranking_score_v1"].min()),
            "mean": float(business_scoring_df["preliminary_ranking_score_v1"].mean()),
            "median": float(business_scoring_df["preliminary_ranking_score_v1"].median()),
            "max": float(business_scoring_df["preliminary_ranking_score_v1"].max()),
        },
        "top_ranking_ready": (
            business_scoring_df[
                business_scoring_df["is_ranking_ready_v1"]
            ]
            .head(50)
            [
                [
                    "business_id",
                    "business_name",
                    "city",
                    "state",
                    "canonical_dish_name_v2",
                    "mention_count",
                    "review_count",
                    "positive_ratio",
                    "negative_ratio",
                    "confidence_weighted_sentiment",
                    "bayesian_sentiment_score",
                    "total_signal_weight",
                    "business_dish_evidence_tier",
                    "preliminary_ranking_score_v1",
                    "preliminary_candidate_tier_v1",
                ]
            ]
            .to_dict(orient="records")
        ),
    },

    "global_scoring": {
        "total_dishes": int(len(global_scoring_df)),
        "evidence_tier_counts": {
            str(k): int(v)
            for k, v in global_scoring_df["global_evidence_tier"].value_counts().to_dict().items()
        },
        "hard_noise_dishes": int(global_scoring_df["is_hard_noise_dish_name"].sum()),
        "possible_beverage_dishes": int(global_scoring_df["is_possible_beverage"].sum()),
        "top_global_scored": (
            global_scoring_df
            .head(50)
            [
                [
                    "canonical_dish_name_v2",
                    "mention_count",
                    "review_count",
                    "business_count",
                    "positive_ratio",
                    "negative_ratio",
                    "confidence_weighted_sentiment",
                    "global_evidence_tier",
                    "global_ranking_signal_score_v1",
                ]
            ]
            .to_dict(orient="records")
        ),
    }
}

print(json.dumps(scoring_summary, indent=2, ensure_ascii=False)[:7000])

print("\nCandidate tiers:")
display(business_scoring_df["preliminary_candidate_tier_v1"].value_counts())

print("\nEvidence tiers negocio-plato:")
display(business_scoring_df["business_dish_evidence_tier"].value_counts())

print("\nTop ranking ready:")
display(
    business_scoring_df[
        business_scoring_df["is_ranking_ready_v1"]
    ]
    [
        [
            "business_name",
            "city",
            "state",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "positive_ratio",
            "negative_ratio",
            "bayesian_sentiment_score",
            "total_signal_weight",
            "business_dish_evidence_tier",
            "preliminary_ranking_score_v1",
            "preliminary_candidate_tier_v1",
        ]
    ]
    .head(50)
)

print("\nPares rechazados por ruido:")
display(
    business_scoring_df[
        business_scoring_df["is_hard_noise_dish_name"]
    ]
    [
        [
            "business_name",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "dish_name_quality_flags",
        ]
    ]
    .head(50)
)

{
  "notebook": "10_dish_signal_aggregation",
  "version": "signal_scoring_v1",
  "source_mentions_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_with_sentiment_hybrid_v1.jsonl",
  "input": {
    "total_mentions": 94932,
    "unique_reviews": 42461,
    "unique_businesses": 4088,
    "unique_dishes": 9937,
    "sentiment_counts": {
      "neutral": 45366,
      "positive": 43142,
      "negative": 6424
    },
    "reliability_counts": {
      "low": 44426,
      "medium": 25495,
      "high": 25011
    },
    "signal_weight": {
      "min": 0.020311710990965366,
      "mean": 0.37781376200795724,
      "median": 0.3624973218077422,
      "max": 0.9497699350118637
    },
    "weighted_sentiment_value": {
      "min": -1.0,
      "mean": 0.3117034681140184,
      "median": 0.041755,
      "max": 1.0
    }
  },
  "business_dish_scoring": {
    "total_pairs": 31036,
    "ranking_ready_v1": 3841,
    "candidate_tier_counts": {
      "not_ready": 28147,


preliminary_candidate_tier_v1
not_ready              28147
weak_candidate          1223
promising_candidate     1021
strong_candidate         645
Name: count, dtype: int64


Evidence tiers negocio-plato:


business_dish_evidence_tier
weak        27194
emerging     1440
solid        1406
strong        996
Name: count, dtype: int64


Top ranking ready:


,business_name,city,state,canonical_dish_name_v2,mention_count,review_count,positive_ratio,negative_ratio,bayesian_sentiment_score,total_signal_weight,business_dish_evidence_tier,preliminary_ranking_score_v1,preliminary_candidate_tier_v1
0,The Little Lamb,Clearwater,FL,burger,15,13,0.866667,0.000000,0.600349,8.924912,strong,88.805930,strong_candidate
1,East Beach Grill,Santa Barbara,CA,pancake,24,14,0.791667,0.000000,0.665284,14.285097,strong,88.768158,strong_candidate
2,Three Muses,New Orleans,LA,steak,32,26,0.781250,0.000000,0.663566,17.626015,strong,88.061879,strong_candidate
3,Sushi Ushi,Valrico,FL,sushi,53,28,0.716981,0.000000,0.694509,27.414415,strong,87.580242,strong_candidate
4,4 Rivers Smokehouse,Tampa Bay,FL,brisket,16,13,0.750000,0.000000,0.606483,10.260428,strong,87.560215,strong_candidate
5,Goody's Soda Fountain & Candy Store,Boise,ID,ice cream,30,20,0.666667,0.000000,0.657213,16.228838,strong,87.439295,strong_candidate
6,Backspace Bar & Kitchen,New Orleans,LA,burger,18,14,0.777778,0.000000,0.640074,10.398027,strong,87.436818,strong_candidate
7,Barbecue and Bourbon,Speedway,IN,pulled pork,12,12,0.833333,0.000000,0.553283,8.251728,strong,87.340456,strong_candidate
8,Empress Garden,Philadelphia,PA,pancake,15,14,0.866667,0.000000,0.602247,8.365439,strong,87.238194,strong_candidate
9,Plaza Deli,Santa Barbara,CA,sandwich,19,16,0.789474,0.000000,0.607592,10.464968,strong,87.223795,strong_candidate



Pares rechazados por ruido:


,business_name,canonical_dish_name_v2,mention_count,review_count,dish_name_quality_flags
30230,Burger Republic,96 burger,4,4,[contains_digit]
30231,500 Degrees,500 burger,5,4,[contains_digit]
30232,Mr. B's Bistro,soup 1-1-1,4,4,[contains_digit]
30233,Livery - Indianapolis,tres leche cake with housemade coffee ice cream,2,2,[too_many_words]
30234,East Coast Pizza,salad : the size was,2,2,"[bad_end_word, contains_suspicious_words, suspicious_punctuation]"
30235,East Coast Pizza,up,2,2,"[exact_noise_name, too_short_name]"
30236,Burger Republic,4 cheese garlic burger,2,2,[contains_digit]
30237,Seasons Cafe & Bakery,mi,2,2,[too_short_name]
30238,Luke,french 75,2,2,[contains_digit]
30239,Mr. B's Bistro,1-1-1,2,2,[contains_digit]


In [21]:
# ============================================================
# 21. Muestras de revisión para ranking
# ============================================================

ranking_ready_sample_df = (
    business_scoring_df[
        business_scoring_df["is_ranking_ready_v1"]
    ]
    .head(500)
    .copy()
)

strong_candidates_sample_df = (
    business_scoring_df[
        business_scoring_df["preliminary_candidate_tier_v1"] == "strong_candidate"
    ]
    .head(500)
    .copy()
)

promising_candidates_sample_df = (
    business_scoring_df[
        business_scoring_df["preliminary_candidate_tier_v1"] == "promising_candidate"
    ]
    .head(500)
    .copy()
)

emerging_candidates_sample_df = (
    business_scoring_df[
        (business_scoring_df["business_dish_evidence_tier"] == "emerging")
        & (business_scoring_df["is_ranking_ready_v1"])
    ]
    .head(500)
    .copy()
)

noise_rejected_sample_df = (
    business_scoring_df[
        business_scoring_df["is_hard_noise_dish_name"]
    ]
    .head(500)
    .copy()
)

top_global_scored_sample_df = global_scoring_df.head(500).copy()

print("ranking_ready_sample_df:", len(ranking_ready_sample_df))
print("strong_candidates_sample_df:", len(strong_candidates_sample_df))
print("promising_candidates_sample_df:", len(promising_candidates_sample_df))
print("emerging_candidates_sample_df:", len(emerging_candidates_sample_df))
print("noise_rejected_sample_df:", len(noise_rejected_sample_df))
print("top_global_scored_sample_df:", len(top_global_scored_sample_df))

ranking_ready_sample_df: 500
strong_candidates_sample_df: 500
promising_candidates_sample_df: 500
emerging_candidates_sample_df: 500
noise_rejected_sample_df: 500
top_global_scored_sample_df: 500


In [22]:
# ============================================================
# 22. Guardar outputs de scoring
# ============================================================

business_ranking_candidates_path = SIGNALS_DIR / "dish_business_ranking_candidates_v1.csv"
global_ranking_signals_path = SIGNALS_DIR / "dish_global_ranking_signals_v1.csv"
scoring_summary_path = ARTIFACTS_DIR / "dish_signal_scoring_summary_v1.json"

ranking_ready_sample_path = SAMPLES_DIR / "dish_business_ranking_ready_sample_v1.csv"
strong_candidates_sample_path = SAMPLES_DIR / "dish_business_strong_candidates_sample_v1.csv"
promising_candidates_sample_path = SAMPLES_DIR / "dish_business_promising_candidates_sample_v1.csv"
emerging_candidates_sample_path = SAMPLES_DIR / "dish_business_emerging_candidates_sample_v1.csv"
noise_rejected_sample_path = SAMPLES_DIR / "dish_business_noise_rejected_sample_v1.csv"
top_global_scored_sample_path = SAMPLES_DIR / "dish_global_scored_sample_v1.csv"

business_scoring_df.to_csv(business_ranking_candidates_path, index=False)
global_scoring_df.to_csv(global_ranking_signals_path, index=False)

ranking_ready_sample_df.to_csv(ranking_ready_sample_path, index=False)
strong_candidates_sample_df.to_csv(strong_candidates_sample_path, index=False)
promising_candidates_sample_df.to_csv(promising_candidates_sample_path, index=False)
emerging_candidates_sample_df.to_csv(emerging_candidates_sample_path, index=False)
noise_rejected_sample_df.to_csv(noise_rejected_sample_path, index=False)
top_global_scored_sample_df.to_csv(top_global_scored_sample_path, index=False)

scoring_summary["outputs"] = {
    "business_ranking_candidates": str(business_ranking_candidates_path),
    "global_ranking_signals": str(global_ranking_signals_path),
    "scoring_summary": str(scoring_summary_path),
    "ranking_ready_sample": str(ranking_ready_sample_path),
    "strong_candidates_sample": str(strong_candidates_sample_path),
    "promising_candidates_sample": str(promising_candidates_sample_path),
    "emerging_candidates_sample": str(emerging_candidates_sample_path),
    "noise_rejected_sample": str(noise_rejected_sample_path),
    "top_global_scored_sample": str(top_global_scored_sample_path),
}

with scoring_summary_path.open("w", encoding="utf-8") as f:
    json.dump(scoring_summary, f, indent=2, ensure_ascii=False)

print("Outputs de scoring guardados:")
print("business_ranking_candidates_path:", business_ranking_candidates_path)
print("global_ranking_signals_path:", global_ranking_signals_path)
print("scoring_summary_path:", scoring_summary_path)

Outputs de scoring guardados:
business_ranking_candidates_path: /kaggle/working/hidden_gems_ai/outputs/dish_signal_aggregation/signals/dish_business_ranking_candidates_v1.csv
global_ranking_signals_path: /kaggle/working/hidden_gems_ai/outputs/dish_signal_aggregation/signals/dish_global_ranking_signals_v1.csv
scoring_summary_path: /kaggle/working/hidden_gems_ai/outputs/dish_signal_aggregation/artifacts/dish_signal_scoring_summary_v1.json


In [23]:
# ============================================================
# 23. Comprimir outputs del módulo 10
# ============================================================

import shutil

zip_base = Path("/kaggle/working") / "dish_signal_aggregation_outputs_v1"

if zip_base.with_suffix(".zip").exists():
    zip_base.with_suffix(".zip").unlink()

shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

print("ZIP generado:")
print(str(zip_base) + ".zip")

ZIP generado:
/kaggle/working/dish_signal_aggregation_outputs_v1.zip


## Cierre del Notebook 10

En este notebook se ha construido la capa de agregación de señales gastronómicas.

El flujo realizado ha sido:

1. Carga de menciones con sentimiento.
2. Creación de pesos de fiabilidad.
3. Agregación global por plato.
4. Agregación por negocio + plato.
5. Control de calidad de nombres de platos.
6. Scoring conservador para ranking.
7. Generación de candidatos preliminares.

Archivos principales generados:

- `dish_global_signals_v1.csv`
- `dish_business_signals_v1.csv`
- `dish_business_ranking_candidates_v1.csv`
- `dish_global_ranking_signals_v1.csv`
- `dish_signal_aggregation_summary_v1.json`
- `dish_signal_scoring_summary_v1.json`

El archivo principal para el siguiente notebook será:

`dish_business_ranking_candidates_v1.csv`

Este archivo servirá como entrada para:

`11_hidden_gems_ranking_v1.ipynb`